# NB05 — Mine Proximity × CWM Associations

**Notebook:** microbeatlas_cwm / NB05  
**Exposure:** Mine proximity (Mindat, 157K localities; 3 operationalizations)  
**KOs tested:** All 6,557 CWM KOs  
**Levels:** L0–L5 (mine distance is the exposure, not a control)  

**Three operationalizations:**
1. `log_prox` = −log₁₀(mine_dist_km + 0.1) — higher = closer
2. `log_dist` = log₁₀(mine_dist_km) — higher = farther
3. `binary` = 1 if <10 km, 0 if >50 km (drop middle zone)

**Commodity stratification:** nearest-mine element type (Fe, Cu, Pb, Zn, Ni) from `mine_elements` column — tests whether proximity to commodity-specific mines predicts corresponding functional CWM shifts.


In [1]:
import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H
apply_style()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist
import patsy
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score, KFold
from pathlib import Path

DATA = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data')
FIGS = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/figures')
FIGS.mkdir(exist_ok=True)


In [2]:
# Load all base tables
nb00     = pd.read_parquet(DATA / 'nb00_thinned_samples.parquet')
combined = pd.read_parquet(DATA / 'nb02_combined_metals.parquet')
mine     = pd.read_parquet(DATA / 'nb02_mine_dist.parquet')
soil     = pd.read_parquet(DATA / 'nb02_soil_props.parquet')
lith     = pd.read_parquet(DATA / 'nb02_lithology.parquet')
wclim    = pd.read_parquet(DATA / 'nb02_worldclim_seasonal.parquet')
gc       = pd.read_parquet(DATA / 'nb01_genus_counts.parquet')
gp       = pd.read_parquet(DATA / 'nb02_genus_phylum.parquet')

# NB02 USA L1 hits for reverse analysis
usa_fdr    = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
usa_L1_hits = usa_fdr[(usa_fdr['level']=='L1') & (usa_fdr['q_bh']<0.05)]

# Build base DataFrame
base = (mine
        .merge(nb00[['sample_id','lat','lon','ph','olm_soil_ph_0cm_H2O','ph_soilgrids',
                      'era5_mean_2m_air_temp_k','era5_total_precipitation_mm',
                      'lights_radiance_nanow_cm2_sr','dem_elevation_m']], on='sample_id', how='left')
        .merge(soil[['sample_id','clay_pct','som_pct','bulk_density']], on='sample_id', how='left')
        .merge(lith[['sample_id','lith_dist_deg']], on='sample_id', how='left')
        .merge(wclim[['sample_id','temp_seasonality','precip_seasonality']], on='sample_id', how='left'))

ph_raw_b = pd.to_numeric(base['ph'], errors='coerce')
ph_olm_b = pd.to_numeric(base['olm_soil_ph_0cm_H2O'], errors='coerce') / 10.0
base['ph_best'] = ph_raw_b.where(ph_raw_b.notna(),
                  ph_olm_b.where(ph_olm_b.notna(), base['ph_soilgrids']))

# Compute Shannon diversity and phylum RAs
tot = gc.groupby('sample_id')['genus_count'].sum().rename('total')
gc2 = gc.join(tot, on='sample_id')
gc2['ra'] = gc2['genus_count'] / gc2['total']
gc2['h'] = -gc2['ra'] * np.log(gc2['ra'].clip(lower=1e-12))
shannon = gc2.groupby('sample_id')['h'].sum().rename('shannon')
gc2 = gc2.merge(gp, on='genus_lower', how='left')
gc2['phylum_lower'] = gc2['phylum_lower'].fillna('unknown')
phyl_ra = gc2.groupby(['sample_id','phylum_lower'])['ra'].sum().reset_index()
top_phyla = phyl_ra.groupby('phylum_lower')['ra'].mean().nlargest(8).index.tolist()
phyl_wide = (phyl_ra[phyl_ra['phylum_lower'].isin(top_phyla)]
             .pivot_table(index='sample_id', columns='phylum_lower', values='ra', fill_value=0.0))
phyl_wide.columns = [f'phyl_{c}' for c in phyl_wide.columns]
base = (base.merge(shannon.reset_index(), on='sample_id', how='left')
            .merge(phyl_wide.reset_index(), on='sample_id', how='left'))
phyl_cols = [c for c in base.columns if c.startswith('phyl_')]
base[phyl_cols] = base[phyl_cols].fillna(0.0)

print(f'base: {base.shape}')
print(f'mine_dist_km: {base["mine_dist_km"].describe()["50%"]:.1f} km median')


base: (4884, 28)
mine_dist_km: 49.9 km median


In [3]:
# Elevation relative to nearest mine (Mindat lat/lon -> DEM proxy)
# elev_rel > 0: sample is uphill from nearest mine; < 0: downhill (higher contamination risk).
# Mine elevation approximated from nearest thinned MA sample's dem_elevation_m.
from scipy.spatial import KDTree as _KDTree
from pathlib import Path as _Path

MINDAT_PATH = _Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/mindat.csv')
print('Loading Mindat for elevation computation...')
md_coords = pd.read_csv(MINDAT_PATH, low_memory=False, usecols=['latitude','longitude'])
md_coords['latitude']  = pd.to_numeric(md_coords['latitude'],  errors='coerce')
md_coords['longitude'] = pd.to_numeric(md_coords['longitude'], errors='coerce')
md_coords = md_coords.dropna()
md_coords = md_coords[
    md_coords['latitude'].between(-90,90) & md_coords['longitude'].between(-180,180)]
print(f'Mindat: {len(md_coords):,} localities with lat/lon')

# KDTree: sample -> nearest mine coordinates
tree_mine = _KDTree(md_coords[['latitude','longitude']].values)
_, mine_idxs = tree_mine.query(base[['lat','lon']].values, k=1)
base['mine_lat'] = md_coords['latitude'].values[mine_idxs]
base['mine_lon'] = md_coords['longitude'].values[mine_idxs]

# Approximate mine DEM elevation via nearest thinned MA sample
nb00_elev = nb00[['lat','lon','dem_elevation_m']].dropna().reset_index(drop=True)
tree_samp = _KDTree(nb00_elev[['lat','lon']].values)
mine_locs = base[['mine_lat','mine_lon']].values
_, samp_idxs = tree_samp.query(mine_locs, k=1)
base['mine_elev_approx'] = nb00_elev['dem_elevation_m'].values[samp_idxs]

# elev_rel: sample elevation - mine elevation (positive = uphill from mine)
base['elev_rel'] = base['dem_elevation_m'] - base['mine_elev_approx']
print(f'elev_rel: mean={base["elev_rel"].mean():.1f} m, sd={base["elev_rel"].std():.1f} m')
pct_downhill = 100 * (base['elev_rel'] < 0).mean()
print(f'Downhill from nearest mine (elev_rel < 0): {pct_downhill:.1f}% of samples')


Loading Mindat for elevation computation...


Mindat: 157,301 localities with lat/lon
elev_rel: mean=-39.1 m, sd=367.0 m
Downhill from nearest mine (elev_rel < 0): 23.4% of samples


In [4]:
# Three mine-proximity operationalizations
base['log_prox'] = -np.log10(base['mine_dist_km'] + 0.1)   # higher = closer
base['log_dist'] = np.log10(base['mine_dist_km'])           # higher = farther
base['binary']   = np.where(base['mine_dist_km'] < 10, 1.0,
                     np.where(base['mine_dist_km'] > 50, 0.0, np.nan))

n_near   = (base['mine_dist_km'] < 10).sum()
n_far    = (base['mine_dist_km'] > 50).sum()
n_middle = base['binary'].isna().sum()
print(f'Binary: near (<10 km) = {n_near}, far (>50 km) = {n_far}, dropped (10-50 km) = {n_middle}')

OPERATIONALIZATIONS = {
    'log_prox': 'log₁₀ proximity (−log₁₀(dist+0.1))',
    'log_dist': 'log₁₀ distance (farther = higher)',
    'binary':   'binary near (<10 km) vs far (>50 km)',
}
print('\nDistribution:')
for op in OPERATIONALIZATIONS:
    v = base[op].dropna()
    print(f'  {op}: n={len(v)}, mean={v.mean():.3f}, sd={v.std():.3f}')


Binary: near (<10 km) = 498, far (>50 km) = 2440, dropped (10-50 km) = 1946

Distribution:
  log_prox: n=4884, mean=-1.653, sd=0.508
  log_dist: n=4884, mean=1.651, sd=0.513
  binary: n=2938, mean=0.170, sd=0.375


In [5]:
# Pivot CWM to wide matrix (all samples × all KOs)
print('Loading CWM long format...')
cwm_long = pd.read_parquet(DATA / 'nb01_cwm.parquet')
print(f'CWM long: {len(cwm_long):,} rows')

# Align to base sample_ids
cwm_sids = set(cwm_long['sample_id'])
base_sids = set(base['sample_id'])
print(f'Samples in CWM ∩ base: {len(cwm_sids & base_sids)}')

print('Pivoting CWM to wide...')
cwm_wide = cwm_long.pivot_table(index='sample_id', columns='ko_id', values='cwm', fill_value=0.0)
base_indexed = base.set_index('sample_id')
cwm_wide = cwm_wide.reindex(base_indexed.index).fillna(0.0)
ko_ids = cwm_wide.columns.tolist()
print(f'CWM wide: {cwm_wide.shape}  (samples × KOs)')


Loading CWM long format...


CWM long: 23,233,790 rows


Samples in CWM ∩ base: 4868
Pivoting CWM to wide...


CWM wide: (4884, 6557)  (samples × KOs)


In [6]:
# Vectorized FWL
def fwl_all_kos(X, Y, Z):
    n, p = Z.shape
    coef_x, *_ = np.linalg.lstsq(Z, X, rcond=None)
    Mx = X - Z @ coef_x
    coef_y, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    My = Y - Z @ coef_y
    MxMx = Mx @ Mx
    betas = My.T @ Mx / MxMx
    resid = My - np.outer(Mx, betas)
    dof = max(n - p - 1, 1)
    sigma2 = (resid ** 2).sum(axis=0) / dof
    se = np.sqrt(sigma2 / MxMx)
    t_stat = betas / np.where(se > 0, se, np.nan)
    return betas, se, t_stat

def build_Z(df, level):
    n = len(df)
    Z = np.ones((n, 1))
    if level == 0:
        return Z

    ph = pd.to_numeric(df['ph_best'], errors='coerce').values
    ph_fill = np.where(np.isfinite(ph), ph, np.nanmedian(ph))
    try:
        ph_dm = patsy.dmatrix('cr(ph_v, df=3)', {'ph_v': ph_fill}, return_type='matrix')[:, 1:]
    except Exception:
        ph_dm = ph_fill.reshape(-1, 1)
    Z = np.hstack([Z, ph_dm])
    if level == 1:
        return Z

    for c in ['clay_pct','som_pct','bulk_density','lith_dist_deg']:
        col = pd.to_numeric(df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col,
                       np.nanmedian(col[np.isfinite(col)]) if np.any(np.isfinite(col)) else 0.0)
        Z = np.hstack([Z, col.reshape(-1, 1)])
    # Elevation relative to nearest mine (topographic drainage control)
    elev_r = pd.to_numeric(df['elev_rel'], errors='coerce').values
    elev_r = np.where(np.isfinite(elev_r), elev_r,
                      np.nanmedian(elev_r[np.isfinite(elev_r)]) if np.any(np.isfinite(elev_r)) else 0.0)
    Z = np.hstack([Z, elev_r.reshape(-1, 1)])
    if level == 2:
        return Z

    # L3: nighttime lights (anthropogenic), NOT mine distance (that's the exposure)
    lights = pd.to_numeric(df['lights_radiance_nanow_cm2_sr'], errors='coerce').values
    lights = np.where(lights > 0, lights, 0.001)
    lights = np.where(np.isfinite(lights), lights, 0.001)
    Z = np.hstack([Z, np.log10(lights).reshape(-1, 1)])
    if level == 3:
        return Z

    for c in ['era5_mean_2m_air_temp_k','era5_total_precipitation_mm',
               'temp_seasonality','precip_seasonality']:
        col = pd.to_numeric(df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col,
                       np.nanmedian(col[np.isfinite(col)]) if np.any(np.isfinite(col)) else 0.0)
        Z = np.hstack([Z, col.reshape(-1, 1)])
    if level == 4:
        return Z

    sh = pd.to_numeric(df['shannon'], errors='coerce').values
    sh = np.where(np.isfinite(sh), sh, np.nanmedian(sh[np.isfinite(sh)]) if np.any(np.isfinite(sh)) else 0.0)
    Z = np.hstack([Z, sh.reshape(-1, 1)])
    for pc in phyl_cols:
        col = pd.to_numeric(df[pc], errors='coerce').fillna(0.0).values
        Z = np.hstack([Z, col.reshape(-1, 1)])
    return Z  # L5


In [7]:
# Forward FWL: generic mine proximity × all KOs × L0–L5
# Run all 3 operationalizations
print('Running forward FWL (generic mine proximity)...')
all_results = []

for op_key in OPERATIONALIZATIONS:
    exposure = base_indexed[op_key].values
    iqr_exp  = np.nanpercentile(exposure[np.isfinite(exposure)], 75) - \
               np.nanpercentile(exposure[np.isfinite(exposure)], 25)

    for level in range(6):  # L0-L5
        Z = build_Z(base_indexed, level)
        cwm_vals = cwm_wide.values
        valid = np.isfinite(exposure) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 50:
            print(f'  {op_key} L{level}: n={n_valid} < 50, skip')
            continue

        betas, se, t_stat = fwl_all_kos(
            exposure[valid], cwm_vals[valid], Z[valid])
        dof = max(n_valid - Z.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)

        for i, ko in enumerate(ko_ids):
            all_results.append({
                'operationalization': op_key, 'level': f'L{level}', 'ko_id': ko,
                'n': n_valid, 'beta': betas[i], 'se': se[i],
                't_stat': t_stat[i], 'p': pvals[i],
                'beta_per_iqr': betas[i] * iqr_exp,
            })
        print(f'  {op_key} L{level}: n={n_valid}')

results_df = pd.DataFrame(all_results)
print(f'Total rows: {len(results_df):,}')


Running forward FWL (generic mine proximity)...


  log_prox L0: n=4884


  log_prox L1: n=4884


  log_prox L2: n=4884


  log_prox L3: n=4884


...

  binary L1: n=2938


  binary L2: n=2938


  binary L3: n=2938


  binary L4: n=2938


  binary L5: n=2938
Total rows: 118,026


In [8]:
# Apply BH-FDR within each operationalization × level
results_df['q_bh'] = np.nan
for (op, lv), grp in results_df.groupby(['operationalization','level']):
    valid_p = np.isfinite(grp['p'].values)
    q = np.full(len(grp), np.nan)
    if valid_p.sum() > 0:
        _, q[valid_p], _, _ = multipletests(grp['p'].values[valid_p], method='fdr_bh')
    results_df.loc[grp.index, 'q_bh'] = q

# Summarize hits
hits_L1 = results_df[(results_df['level']=='L1') & (results_df['q_bh']<0.05)]
print('L1 FDR hits by operationalization:')
for op, grp in hits_L1.groupby('operationalization'):
    print(f'  {op}: {len(grp)} hits, {grp["ko_id"].nunique()} unique KOs')

print('\nHits across levels:')
hits_all = results_df[results_df['q_bh']<0.05]
print(hits_all.groupby(['operationalization','level']).size().unstack(fill_value=0).to_string())

results_df.attrs = {}
results_df.to_parquet(DATA / 'nb05_mine_fwl.parquet', index=False)
print('\nSaved: nb05_mine_fwl.parquet')


L1 FDR hits by operationalization:
  log_dist: 462 hits, 462 unique KOs
  log_prox: 481 hits, 481 unique KOs

Hits across levels:
level                L0   L1   L2   L3   L4  L5
operationalization                             
log_dist            288  462  318  487  242  57
log_prox            299  481  352  496  242  54



Saved: nb05_mine_fwl.parquet


In [9]:
# Commodity-specific stratification: FWL within nearest-mine-element subsets
# Tests ALL 6,557 KOs x L1 -- full genome-wide screen per commodity.
COMMODITY_MAP = {'Cu': 'Cu', 'Pb': 'Pb', 'Zn': 'Zn', 'Ni': 'Ni'}

commodity_results = []
for mine_elem, metal in COMMODITY_MAP.items():
    sub_idx = base_indexed[base_indexed['mine_elements'] == mine_elem].index
    n_sub = len(sub_idx)
    if n_sub < 50:
        print(f'{mine_elem}: n={n_sub} < 50, skip')
        continue

    sub_base = base_indexed.loc[sub_idx]
    exposure = sub_base['log_prox'].values
    iqr_exp = (np.nanpercentile(exposure[np.isfinite(exposure)], 75) -
               np.nanpercentile(exposure[np.isfinite(exposure)], 25))

    cwm_sub = cwm_wide.reindex(sub_idx).fillna(0.0).values   # all KOs
    Z = build_Z(sub_base, 1)   # L1: intercept + pH spline
    valid = np.isfinite(exposure) & np.all(np.isfinite(Z), axis=1)
    n_valid = int(valid.sum())
    if n_valid < 30:
        print(f'{mine_elem}: n_valid={n_valid} < 30, skip')
        continue

    betas, se, t_stat = fwl_all_kos(
        exposure[valid], cwm_sub[valid], Z[valid])
    dof = max(n_valid - Z.shape[1] - 1, 1)
    pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)

    q_full = np.full(len(ko_ids), np.nan)
    valid_p = np.isfinite(pvals)
    if valid_p.sum() > 0:
        _, q_vals, _, _ = multipletests(pvals[valid_p], method='fdr_bh')
        q_full[valid_p] = q_vals

    for i, ko in enumerate(ko_ids):
        commodity_results.append({
            'mine_elem': mine_elem, 'metal': metal, 'ko_id': ko,
            'n': n_valid, 'beta': betas[i], 'se': se[i],
            'p': pvals[i], 'q_bh': q_full[i],
            'beta_per_iqr': betas[i] * iqr_exp,
        })
    n_hits = int((q_full < 0.05).sum())
    print(f'{mine_elem} mines: n={n_valid}, FDR hits = {n_hits}/{len(ko_ids)} (all KOs, L1)')

comm_df = pd.DataFrame(commodity_results)
comm_hits = comm_df[comm_df['q_bh'] < 0.05]
print(f'\nTotal commodity-specific FDR hits: {len(comm_hits)}')

# Overlap with NB02 measured-metal L1 hits
nb02_hits = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
nb02_L1 = nb02_hits[(nb02_hits['level']=='L1') & (nb02_hits['q_bh']<0.05)]
print('\nOverlap: commodity mine hits vs NB02 measured-metal hits:')
for mine_elem, metal in COMMODITY_MAP.items():
    comm_ko = set(comm_hits[comm_hits['mine_elem']==mine_elem]['ko_id'])
    nb02_ko = set(nb02_L1[nb02_L1['metal']==metal]['ko_id'])
    ov = comm_ko & nb02_ko
    pct = 100*len(ov)/max(len(comm_ko),1)
    print(f'  {mine_elem}: {len(comm_ko)} mine hits, {len(nb02_ko)} metal hits, '
          f'overlap={len(ov)} ({pct:.1f}%)')

comm_df.attrs = {}
comm_df.to_parquet(DATA / 'nb05_commodity_fwl_allkos.parquet', index=False)
print('Saved nb05_commodity_fwl_allkos.parquet')


Cu mines: n=693, FDR hits = 0/6557 (all KOs, L1)


Pb mines: n=424, FDR hits = 0/6557 (all KOs, L1)


Zn mines: n=401, FDR hits = 0/6557 (all KOs, L1)
Ni mines: n=232, FDR hits = 0/6557 (all KOs, L1)

Total commodity-specific FDR hits: 0



Overlap: commodity mine hits vs NB02 measured-metal hits:
  Cu: 0 mine hits, 0 metal hits, overlap=0 (0.0%)
  Pb: 0 mine hits, 14 metal hits, overlap=0 (0.0%)
  Zn: 0 mine hits, 76 metal hits, overlap=0 (0.0%)
  Ni: 0 mine hits, 38 metal hits, overlap=0 (0.0%)
Saved nb05_commodity_fwl_allkos.parquet


In [10]:
# Reverse: CWM → log_prox (mine proximity as target), RidgeCV + 5-fold CV
# Use NB02 L1 hit KOs as features (across all metals)
print('Running reverse analysis: CWM → mine proximity...')
hit_ko_set = set(usa_L1_hits['ko_id'].tolist())

X_all = base_indexed['log_prox'].values
valid_all = np.isfinite(X_all)

# Also run pH as positive control (should replicate NB02's pH positive control)
for target_name, target_vals, feature_kos in [
        ('log_prox (mine)', base_indexed['log_prox'].values, list(hit_ko_set)),
        ('pH (positive ctrl)', base_indexed['ph_best'].values,
         usa_fdr[(usa_fdr['level']=='L1') & (usa_fdr['metal'].isin(['As','Zn'])) & (usa_fdr['q_bh']<0.05)]['ko_id'].unique().tolist()[:200]),
    ]:

    target = pd.to_numeric(pd.Series(target_vals, index=base_indexed.index), errors='coerce')
    valid = np.isfinite(target.values)
    feature_kos_avail = [k for k in feature_kos if k in cwm_wide.columns]
    if len(feature_kos_avail) < 5 or valid.sum() < 100:
        print(f'{target_name}: insufficient data, skip')
        continue

    Y_feat = cwm_wide[feature_kos_avail].reindex(base_indexed.index).fillna(0.0).values[valid]
    X_tgt  = target.values[valid]

    ridge = RidgeCV(alphas=np.logspace(-3, 5, 20), cv=5)
    ridge.fit(Y_feat, X_tgt)
    r2_random = ridge.score(Y_feat, X_tgt)

    # 5-fold spatial CV using lat/lon blocks
    lats = pd.to_numeric(base_indexed['lat'], errors='coerce').values[valid]
    lons = pd.to_numeric(base_indexed['lon'], errors='coerce').values[valid]
    from sklearn.cluster import KMeans
    coords = np.column_stack([lats, lons])
    coords_ok = np.isfinite(coords).all(axis=1)
    kf_labels = np.zeros(len(X_tgt), dtype=int)
    if coords_ok.sum() > 50:
        km = KMeans(n_clusters=5, n_init=10, random_state=42)
        km.fit(coords[coords_ok])
        kf_labels[coords_ok] = km.labels_
    r2_spatial = cross_val_score(RidgeCV(alphas=np.logspace(-3,5,20)),
                                  Y_feat, X_tgt, cv=[
                                      (np.where(kf_labels!=k)[0], np.where(kf_labels==k)[0])
                                      for k in range(5)
                                  ], scoring='r2').mean()

    print(f'{target_name} (n={len(X_tgt)}, n_feat={len(feature_kos_avail)}): '
          f'random-CV R²={r2_random:.3f}, spatial-CV R²={r2_spatial:.3f}')


Running reverse analysis: CWM → mine proximity...


log_prox (mine) (n=4884, n_feat=427): random-CV R²=0.000, spatial-CV R²=-0.191


pH (positive ctrl) (n=4844, n_feat=78): random-CV R²=0.103, spatial-CV R²=-0.033


In [11]:
# Figure: hit count across levels for each operationalization
level_order = ['L0','L1','L2','L3','L4','L5']
ops = list(OPERATIONALIZATIONS.keys())

fig, axes = plt.subplots(1, 3, figsize=(FIGW['full'], ROW_H), sharey=False)
for ax, op in zip(axes, ops):
    op_df = results_df[results_df['operationalization']==op]
    hits_by_lv = (op_df[op_df['q_bh']<0.05]
                  .groupby('level').size()
                  .reindex(level_order, fill_value=0))
    ax.bar(level_order, hits_by_lv.values, color=PALETTE[0], edgecolor='k', linewidth=0.5)
    for lv, v in zip(level_order, hits_by_lv.values):
        if v > 0:
            ax.text(lv, v + 0.5, str(int(v)), ha='center', va='bottom', fontsize=8, color='#808080')
    ax.set_xlabel('Causal level')
    ax.set_ylabel('FDR hits (q<0.05)')
    ax.set_title(OPERATIONALIZATIONS[op], fontsize=9)

fig.suptitle('Mine proximity → CWM: FDR hits by causal level and operationalization', y=1.02)
fig.tight_layout()
save(fig, FIGS / 'fig_nb05_mine_hits_by_level')


In [12]:
# Figure: distribution of elev_rel and its relationship to mine proximity
from figure_style import grid_h
fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Panel A: elev_rel histogram
ax = axes[0]
er = base_indexed['elev_rel'].dropna()
ax.hist(er.clip(-500,500), bins=50, color=PALETTE[0], edgecolor='k', linewidth=0.5)
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Sample elev − mine elev (m)')
ax.set_ylabel('Count')
ax.set_title('Elevation relative to nearest mine')
pct_dn = 100*(er < 0).mean()
ax.annotate(f'{pct_dn:.1f}% downhill', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=8)

# Panel B: elev_rel vs log_prox (does being close to a mine mean you are downhill?)
ax = axes[1]
grid_h(ax)
x_plot = base_indexed['log_prox'].values
y_plot = base_indexed['elev_rel'].clip(-500,500).values
ok = np.isfinite(x_plot) & np.isfinite(y_plot)
ax.scatter(x_plot[ok], y_plot[ok], s=2, alpha=0.3, color=PALETTE[0], rasterized=True)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('log proximity (−log₁₀(dist+0.1))')
ax.set_ylabel('Sample elev − mine elev (m)')
ax.set_title('Proximity vs elevation gradient')
from scipy.stats import spearmanr
rho, p = spearmanr(x_plot[ok], y_plot[ok])
ax.annotate(f'ρ={rho:.2f}, p={p:.2e}', xy=(0.05, 0.05), xycoords='axes fraction', fontsize=8)

fig.suptitle('Topographic context of mine proximity', y=1.02)
save(fig, FIGS / 'fig_nb05_elev_rel')


In [13]:
# Figure: overlap heatmap — mine proximity L1 hits vs NB02 metal L1 hits
from figure_style import grid_h
METALS_PLOT = ['As','Cd','Cr','Cu','Ni','Pb','Zn']
ops_plot = ['log_prox','log_dist']
usa_L1_hits_load = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
usa_L1_load = usa_L1_hits_load[(usa_L1_hits_load['level']=='L1') & (usa_L1_hits_load['q_bh']<0.05)]
mine_L1 = results_df[(results_df['level']=='L1') & (results_df['q_bh']<0.05)]

pct_mat = np.zeros((len(ops_plot), len(METALS_PLOT)))
for i, op in enumerate(ops_plot):
    mine_kos = set(mine_L1[mine_L1['operationalization']==op]['ko_id'])
    for j, m in enumerate(METALS_PLOT):
        nb02_kos = set(usa_L1_load[usa_L1_load['metal']==m]['ko_id'])
        ov = mine_kos & nb02_kos
        pct_mat[i,j] = 100*len(ov)/max(len(mine_kos),1)

fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H*0.7))
im = ax.imshow(pct_mat, cmap='YlOrRd', vmin=0, vmax=10, aspect='auto')
plt.colorbar(im, ax=ax, label='% overlap', shrink=0.8)
ax.set_xticks(range(len(METALS_PLOT))); ax.set_xticklabels(METALS_PLOT)
ax.set_yticks(range(len(ops_plot))); ax.set_yticklabels(['log prox','log dist'])
for i in range(pct_mat.shape[0]):
    for j in range(pct_mat.shape[1]):
        ax.text(j, i, f'{pct_mat[i,j]:.1f}%', ha='center', va='center', fontsize=7)
ax.set_xlabel('NB02 measured-metal hits')
ax.set_title('Mine proximity L1 hit overlap with NB02 metal hits')
save(fig, FIGS / 'fig_nb05_mine_metal_overlap')


In [14]:
# Summary
print('=== NB05 Summary ===')
for op in ops:
    op_L1 = results_df[(results_df['operationalization']==op) & (results_df['level']=='L1')]
    n_hits = (op_L1['q_bh'] < 0.05).sum()
    print(f'{op} L1: {n_hits} FDR hits ({op_L1["ko_id"].nunique()} KOs tested)')
print()
print(f'Commodity-specific hits (log_prox within element-specific mine subsets, L1):')
print(f'  Total: {len(comm_hits)}')
if len(comm_hits):
    for mine_elem, grp in comm_hits.groupby('mine_elem'):
        print(f'  {mine_elem}: {len(grp)}')


=== NB05 Summary ===
log_prox L1: 481 FDR hits (6557 KOs tested)
log_dist L1: 462 FDR hits (6557 KOs tested)
binary L1: 0 FDR hits (6557 KOs tested)

Commodity-specific hits (log_prox within element-specific mine subsets, L1):
  Total: 0
